In [ ]:
#!/usr/bin/env python3

import argparse
import csv
import hashlib
import json
import os
import random
import time
from pathlib import Path

import numpy as np
from PIL import Image

import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms


TRAIN_CACHE_PATH = "caches/train_targets_cache.npz"
TEST_CACHE_PATH = "caches/test_targets_cache.npz"
CHECKPOINT_PATH = "checkpoints/view_selector_checkpoint.pt"
OUTPUT_DIR = "view_selector_checkpoint_outputs"

INPUT_MODE = "depth_normal"
IMG_SIZE = 224
BATCH_SIZE = 256
NUM_WORKERS = 8
PIN_MEMORY = True
USE_AMP = True
SEED = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


class Logger:
    def __init__(self, out_dir):
        self.out_dir = Path(out_dir)
        self.out_dir.mkdir(parents=True, exist_ok=True)
        self.path = self.out_dir / f"run_{time.strftime('%Y%m%d_%H%M%S')}.txt"

    def __call__(self, message):
        print(message, flush=True)
        with open(self.path, "a", encoding="utf-8") as file:
            file.write(str(message) + "\n")


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def as_list(value):
    if value is None:
        return None

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, (list, tuple)):
        return list(value)

    return [value]


def pick_key(data, names):
    for name in names:
        if name in data:
            return data[name]

    return None


def clean_path(value):
    if value is None:
        return None

    if isinstance(value, bytes):
        return value.decode("utf-8")

    return str(value)


def infer_group(paths, index):
    for path in paths:
        if path:
            parent = Path(str(path)).parent.name

            if parent:
                return parent

    return str(index // 132)


def build_samples(data, prefix):
    targets = pick_key(
        data,
        (
            f"{prefix}_y",
            f"{prefix}_error",
            f"{prefix}_errors",
            f"{prefix}_dft",
            f"{prefix}_volumetric_dft",
            f"{prefix}_volume_dft",
            f"{prefix}_target",
            f"{prefix}_targets",
        ),
    )

    if targets is None:
        return []

    targets = np.asarray(targets, dtype=np.float32).reshape(-1)

    depth = as_list(
        pick_key(
            data,
            (
                f"{prefix}_d",
                f"{prefix}_depth",
                f"{prefix}_depth_path",
                f"{prefix}_depth_paths",
            ),
        )
    )

    normal = as_list(
        pick_key(
            data,
            (
                f"{prefix}_n",
                f"{prefix}_normal",
                f"{prefix}_normal_path",
                f"{prefix}_normal_paths",
            ),
        )
    )

    rgb = as_list(
        pick_key(
            data,
            (
                f"{prefix}_rgb",
                f"{prefix}_image",
                f"{prefix}_img",
                f"{prefix}_view",
                f"{prefix}_photo",
            ),
        )
    )

    gray = as_list(
        pick_key(
            data,
            (
                f"{prefix}_gray",
                f"{prefix}_grayscale",
            ),
        )
    )

    group = as_list(
        pick_key(
            data,
            (
                f"{prefix}_group",
                f"{prefix}_groups",
                f"{prefix}_folder",
                f"{prefix}_folders",
                f"{prefix}_cad",
                f"{prefix}_model",
            ),
        )
    )

    samples = []

    for index, target in enumerate(targets):
        depth_path = clean_path(depth[index]) if depth is not None and index < len(depth) else None
        normal_path = clean_path(normal[index]) if normal is not None and index < len(normal) else None
        rgb_path = clean_path(rgb[index]) if rgb is not None and index < len(rgb) else None
        gray_path = clean_path(gray[index]) if gray is not None and index < len(gray) else None

        sample_group = (
            str(group[index])
            if group is not None and index < len(group)
            else infer_group((depth_path, normal_path, rgb_path, gray_path), index)
        )

        samples.append(
            {
                "d": depth_path,
                "n": normal_path,
                "rgb": rgb_path,
                "gray": gray_path,
                "y": float(target),
                "group": sample_group,
            }
        )

    return samples


def grouped_split(samples, val_fraction=0.2, seed=42):
    groups = sorted({sample["group"] for sample in samples})
    rng = random.Random(seed)
    rng.shuffle(groups)

    n_val = max(1, int(round(len(groups) * val_fraction))) if len(groups) > 1 else 0
    val_groups = set(groups[:n_val])

    train_samples = [sample for sample in samples if sample["group"] not in val_groups]
    val_samples = [sample for sample in samples if sample["group"] in val_groups]

    if not val_samples:
        n_val_samples = max(1, int(round(len(samples) * val_fraction)))
        val_samples = samples[-n_val_samples:]
        train_samples = samples[:-n_val_samples]

    return train_samples, val_samples


def load_splits(train_cache_path, test_cache_path=None, val_fraction=0.2, seed=42):
    with np.load(train_cache_path, allow_pickle=True) as train_data:
        train_all = build_samples(train_data, "train")
        val_samples = build_samples(train_data, "val")
        test_samples = build_samples(train_data, "test")

    if test_cache_path:
        with np.load(test_cache_path, allow_pickle=True) as test_data:
            external_test = build_samples(test_data, "test")

        if external_test:
            test_samples = external_test

    if not train_all:
        raise RuntimeError("No training samples were found in the cache")

    if not val_samples:
        train_samples, val_samples = grouped_split(
            train_all,
            val_fraction=val_fraction,
            seed=seed,
        )
    else:
        train_samples = train_all

    if not test_samples:
        test_samples = val_samples

    return train_samples, val_samples, test_samples


def mode_available(samples, input_mode):
    if input_mode == "depth":
        return all(sample.get("d") for sample in samples)

    if input_mode == "depth_normal":
        return all(sample.get("d") and sample.get("n") for sample in samples)

    if input_mode == "rgb":
        return all(sample.get("rgb") for sample in samples)

    if input_mode == "gray":
        return all(sample.get("gray") or sample.get("rgb") for sample in samples)

    return False


class ViewDataset(Dataset):
    def __init__(self, samples, input_mode, image_size):
        self.samples = samples
        self.input_mode = input_mode
        self.resize = transforms.Resize((image_size, image_size), antialias=True)
        self.to_tensor = transforms.ToTensor()

    def __len__(self):
        return len(self.samples)

    def load_gray(self, path):
        if not path or not os.path.exists(path):
            raise FileNotFoundError(path)

        with Image.open(path) as image:
            return self.to_tensor(self.resize(image.convert("L")))

    def load_rgb(self, path):
        if not path or not os.path.exists(path):
            raise FileNotFoundError(path)

        with Image.open(path) as image:
            return self.to_tensor(self.resize(image.convert("RGB")))

    def __getitem__(self, index):
        sample = self.samples[index]

        if self.input_mode == "rgb":
            inputs = self.load_rgb(sample["rgb"])

        elif self.input_mode == "gray":
            path = sample["gray"] if sample.get("gray") else sample["rgb"]
            gray = self.load_gray(path)
            inputs = gray.repeat(3, 1, 1)

        elif self.input_mode == "depth":
            inputs = self.load_gray(sample["d"]) * 2.0 - 1.0

        elif self.input_mode == "depth_normal":
            depth = self.load_gray(sample["d"]) * 2.0 - 1.0
            normal = self.load_rgb(sample["n"])
            normal_z = normal[2:3] * 2.0 - 1.0
            inputs = torch.cat((depth, normal_z), dim=0)

        else:
            raise ValueError(f"Unsupported input mode: {self.input_mode}")

        target = torch.tensor(
            [float(np.clip(sample["y"], 0.0, 1.0))],
            dtype=torch.float32,
        )

        return inputs.float(), target, index


def make_loader(samples, input_mode, image_size, batch_size):
    dataset = ViewDataset(
        samples,
        input_mode=input_mode,
        image_size=image_size,
    )

    kwargs = {
        "dataset": dataset,
        "batch_size": batch_size,
        "shuffle": False,
        "num_workers": NUM_WORKERS,
        "pin_memory": PIN_MEMORY and DEVICE.startswith("cuda"),
        "drop_last": False,
    }

    if NUM_WORKERS > 0:
        kwargs["persistent_workers"] = True
        kwargs["prefetch_factor"] = 2

    return DataLoader(**kwargs)


def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)

    return digest.hexdigest()


def load_model_checkpoint(path):
    checkpoint_path = Path(path)

    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"Checkpoint was not found: {checkpoint_path}"
        )

    try:
        model = torch.jit.load(
            str(checkpoint_path),
            map_location=DEVICE,
        )
    except Exception as exc:
        raise RuntimeError(
            "The checkpoint must be a trusted TorchScript model saved with torch.jit.save"
        ) from exc

    model = model.to(DEVICE)
    model.eval()

    return model


def prediction_tensor(output):
    if isinstance(output, torch.Tensor):
        prediction = output

    elif isinstance(output, dict):
        prediction = None

        for key in (
            "prediction",
            "predictions",
            "pred",
            "output",
            "logits",
        ):
            value = output.get(key)

            if isinstance(value, torch.Tensor):
                prediction = value
                break

        if prediction is None:
            raise RuntimeError(
                "The checkpoint output dictionary has no prediction tensor"
            )

    elif isinstance(output, (list, tuple)):
        prediction = next(
            (
                value
                for value in output
                if isinstance(value, torch.Tensor)
            ),
            None,
        )

        if prediction is None:
            raise RuntimeError(
                "The checkpoint output has no prediction tensor"
            )

    else:
        raise RuntimeError(
            f"Unsupported checkpoint output type: {type(output).__name__}"
        )

    if prediction.ndim == 1:
        prediction = prediction.unsqueeze(1)

    if prediction.ndim != 2 or prediction.shape[1] != 1:
        raise RuntimeError(
            "Expected one scalar prediction per image, "
            f"received {tuple(prediction.shape)}"
        )

    return prediction


def r2_score_np(targets, predictions):
    targets = np.asarray(targets, dtype=np.float64).reshape(-1)
    predictions = np.asarray(predictions, dtype=np.float64).reshape(-1)

    residual_sum = float(
        np.sum((targets - predictions) ** 2)
    )

    total_sum = float(
        np.sum((targets - np.mean(targets)) ** 2)
    )

    if total_sum <= 0:
        return float("nan")

    return 1.0 - residual_sum / total_sum


def mse_np(targets, predictions):
    targets = np.asarray(targets, dtype=np.float64).reshape(-1)
    predictions = np.asarray(predictions, dtype=np.float64).reshape(-1)

    return float(
        np.mean((targets - predictions) ** 2)
    )


def mae_np(targets, predictions):
    targets = np.asarray(targets, dtype=np.float64).reshape(-1)
    predictions = np.asarray(predictions, dtype=np.float64).reshape(-1)

    return float(
        np.mean(np.abs(targets - predictions))
    )


@torch.inference_mode()
def evaluate(model, loader):
    targets = []
    predictions = []
    indices = []

    amp_enabled = USE_AMP and DEVICE.startswith("cuda")

    for inputs, target, index in loader:
        inputs = inputs.to(
            DEVICE,
            non_blocking=True,
        )

        with torch.autocast(
            device_type="cuda" if DEVICE.startswith("cuda") else "cpu",
            dtype=torch.float16 if DEVICE.startswith("cuda") else torch.bfloat16,
            enabled=amp_enabled,
        ):
            output = model(inputs)
            prediction = prediction_tensor(output)

        targets.append(target.numpy())
        predictions.append(
            prediction.float().cpu().numpy()
        )
        indices.append(index.numpy())

    target_array = np.concatenate(
        targets,
        axis=0,
    ).reshape(-1)

    prediction_array = np.concatenate(
        predictions,
        axis=0,
    ).reshape(-1)

    index_array = np.concatenate(
        indices,
        axis=0,
    ).reshape(-1)

    order = np.argsort(index_array)

    target_array = target_array[order]
    prediction_array = prediction_array[order]
    index_array = index_array[order]

    return {
        "MSE": mse_np(
            target_array,
            prediction_array,
        ),
        "MAE": mae_np(
            target_array,
            prediction_array,
        ),
        "R2": r2_score_np(
            target_array,
            prediction_array,
        ),
        "y": target_array,
        "pred": prediction_array,
        "index": index_array,
    }


def write_metrics(path, rows):
    fields = (
        "split",
        "samples",
        "MSE",
        "MAE",
        "R2",
    )

    with open(
        path,
        "w",
        newline="",
        encoding="utf-8",
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=fields,
        )
        writer.writeheader()
        writer.writerows(rows)


def write_ranked_predictions(path, samples, result):
    rows = []

    for index, target, prediction in zip(
        result["index"],
        result["y"],
        result["pred"],
    ):
        sample = samples[int(index)]

        rows.append(
            {
                "index": int(index),
                "group": sample["group"],
                "target": float(target),
                "prediction": float(prediction),
            }
        )

    rows.sort(
        key=lambda row: (
            row["group"],
            row["prediction"],
        )
    )

    group_rank = {}

    for row in rows:
        group = row["group"]
        group_rank[group] = group_rank.get(group, 0) + 1
        row["predicted_rank"] = group_rank[group]

    fields = (
        "index",
        "group",
        "predicted_rank",
        "target",
        "prediction",
    )

    with open(
        path,
        "w",
        newline="",
        encoding="utf-8",
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=fields,
        )
        writer.writeheader()
        writer.writerows(rows)


def main():
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--train-cache",
        default=TRAIN_CACHE_PATH,
    )

    parser.add_argument(
        "--test-cache",
        default=TEST_CACHE_PATH,
    )

    parser.add_argument(
        "--checkpoint",
        default=CHECKPOINT_PATH,
    )


    parser.add_argument(
        "--output-dir",
        default=OUTPUT_DIR,
    )

    parser.add_argument(
        "--input-mode",
        choices=(
            "gray",
            "rgb",
            "depth",
            "depth_normal",
        ),
        default=INPUT_MODE,
    )

    parser.add_argument(
        "--image-size",
        type=int,
        default=IMG_SIZE,
    )

    parser.add_argument(
        "--batch-size",
        type=int,
        default=BATCH_SIZE,
    )

    parser.add_argument(
        "--seed",
        type=int,
        default=SEED,
    )

    parser.add_argument(
        "--val-fraction",
        type=float,
        default=0.20,
    )


    args = parser.parse_args()

    seed_all(args.seed)

    output_dir = Path(args.output_dir)
    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    logger = Logger(output_dir)


    checkpoint_hash = sha256_file(
        args.checkpoint
    )

    logger(f"device={DEVICE}")
    logger(f"checkpoint_sha256={checkpoint_hash}")
    logger(f"input_mode={args.input_mode}")

    splits = load_splits(
        args.train_cache,
        args.test_cache,
        val_fraction=args.val_fraction,
        seed=args.seed,
    )

    train_samples, val_samples, test_samples = splits
    all_samples = (
        train_samples
        + val_samples
        + test_samples
    )

    if not mode_available(
        all_samples,
        args.input_mode,
    ):
        raise RuntimeError(
            "The selected input mode is not available "
            f"for every sample: {args.input_mode}"
        )

    model = load_model_checkpoint(
        args.checkpoint
    )

    loaders = {
        "train": make_loader(
            train_samples,
            args.input_mode,
            args.image_size,
            args.batch_size,
        ),
        "val": make_loader(
            val_samples,
            args.input_mode,
            args.image_size,
            args.batch_size,
        ),
        "test": make_loader(
            test_samples,
            args.input_mode,
            args.image_size,
            args.batch_size,
        ),
    }

    sample_sets = {
        "train": train_samples,
        "val": val_samples,
        "test": test_samples,
    }

    results = {}
    metric_rows = []

    for split, loader in loaders.items():
        result = evaluate(
            model,
            loader,
        )

        results[split] = result

        metric_rows.append(
            {
                "split": split,
                "samples": len(sample_sets[split]),
                "MSE": result["MSE"],
                "MAE": result["MAE"],
                "R2": result["R2"],
            }
        )

        logger(
            f"{split}: "
            f"n={len(sample_sets[split])}, "
            f"MSE={result['MSE']:.6f}, "
            f"MAE={result['MAE']:.6f}, "
            f"R2={result['R2']:.6f}"
        )

    write_metrics(
        output_dir / "checkpoint_metrics.csv",
        metric_rows,
    )

    write_ranked_predictions(
        output_dir / "ranked_test_predictions.csv",
        test_samples,
        results["test"],
    )

    np.savez(
        output_dir / "checkpoint_predictions.npz",
        train_y=results["train"]["y"],
        train_pred=results["train"]["pred"],
        val_y=results["val"]["y"],
        val_pred=results["val"]["pred"],
        test_y=results["test"]["y"],
        test_pred=results["test"]["pred"],
    )

    run_metadata = {
        "checkpoint_sha256": checkpoint_hash,
        "input_mode": args.input_mode,
        "image_size": args.image_size,
        "batch_size": args.batch_size,
        "device": DEVICE,
        "metrics": metric_rows,
    }

    with open(
        output_dir / "run_metadata.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            run_metadata,
            file,
            indent=2,
        )

    logger("done")


if __name__ == "__main__":
    main()
